In [1]:
import pandas as pd
import geopandas as gpd
import zipfile
import os

In [2]:
zip_path = "../data/HMA_LS_Cat_point_v02.0.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside ZIP:")
    for file in z.namelist():
        print(file)

Files inside ZIP:
HMA_LS_Cat_point_v02.0.cpg
HMA_LS_Cat_point_v02.0.dbf
HMA_LS_Cat_point_v02.0.prj
HMA_LS_Cat_point_v02.0.sbn
HMA_LS_Cat_point_v02.0.sbx
HMA_LS_Cat_point_v02.0.shp
HMA_LS_Cat_point_v02.0.shp.xml
HMA_LS_Cat_point_v02.0.shx
HMA_LS_Cat_point_v02.0.xml


In [4]:
extract_folder = "../data/HMA_LS_Cat_point_v02.0"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_folder)

print("Dataset extracted successfully.")
print(os.listdir(extract_folder))

Dataset extracted successfully.
['HMA_LS_Cat_point_v02.0.cpg', 'HMA_LS_Cat_point_v02.0.dbf', 'HMA_LS_Cat_point_v02.0.prj', 'HMA_LS_Cat_point_v02.0.sbn', 'HMA_LS_Cat_point_v02.0.sbx', 'HMA_LS_Cat_point_v02.0.shp', 'HMA_LS_Cat_point_v02.0.shp.xml', 'HMA_LS_Cat_point_v02.0.shx', 'HMA_LS_Cat_point_v02.0.xml']


In [5]:
shp_path = "../data/HMA_LS_Cat_point_v02.0/HMA_LS_Cat_point_v02.0.shp"

events = gpd.read_file(shp_path)

print("Shape:", events.shape)
print("\nColumns:")
print(events.columns.tolist())


Shape: (2801, 30)

Columns:
['src_name', 'src_link', 'ev_id', 'ev_date', 'ev_time', 'ev_title', 'ev_desc', 'loc_desc', 'loc_accu', 'ls_cat', 'ls_trig', 'ls_size', 'ls_setting', 'fatalities', 'injuries', 'storm_name', 'photo_link', 'comments', 'ev_imp_src', 'ev_imp_id', 'latitude', 'longitude', 'ctry_name', 'ctry_code', 'div_name', 'gaz_point', 'gaz_dist', 'sub_date', 'edit_date', 'geometry']


In [6]:
print(events.isna().sum().sort_values(ascending=False))

storm_name    2769
comments      2645
photo_link    2479
ev_time       1768
ev_imp_id      628
loc_desc        26
ev_desc         11
src_link         9
sub_date         2
loc_accu         1
div_name         1
ctry_code        1
ctry_name        1
edit_date        0
gaz_dist         0
longitude        0
latitude         0
gaz_point        0
src_name         0
ev_imp_src       0
injuries         0
fatalities       0
ls_setting       0
ls_size          0
ls_trig          0
ls_cat           0
ev_title         0
ev_date          0
ev_id            0
geometry         0
dtype: int64


In [7]:
print(events["ev_date"].head(10))

events["ev_date"] = pd.to_datetime(events["ev_date"], errors="coerce")

print("\nMissing dates after conversion:", events["ev_date"].isna().sum())
print("Earliest event:", events["ev_date"].min())
print("Latest event:", events["ev_date"].max())

0   2008-08-01
1   2009-07-31
2   2008-11-01
3   2008-11-01
4   2009-07-22
5   2009-07-02
6   2009-07-02
7   2009-07-22
8   2009-07-25
9   2009-07-01
Name: ev_date, dtype: datetime64[ms]

Missing dates after conversion: 0
Earliest event: 1990-01-28 00:00:00
Latest event: 2018-12-31 00:00:00


In [8]:
print(events["ctry_name"].value_counts().head(20))
print("\nDivisions/states:")
print(events["div_name"].value_counts().head(30))

ctry_name
India           1349
Nepal            586
China            339
Pakistan         181
Bangladesh        91
Vietnam           65
Kyrgyzstan        64
Myanmar           43
Tajikistan        30
Bhutan            26
Afghanistan       17
Kazakhstan         3
Uzbekistan         2
Thailand           2
Laos               1
Turkmenistan       1
Name: count, dtype: int64

Divisions/states:
div_name
Jammu and Kashmir      295
Uttaranchal            221
Himachal Pradesh       208
Manipur                102
Gandaki                 96
Assam                   94
Nagaland                91
West Bengal             88
Bagmati                 87
Sichuan                 86
Arunachal Pradesh       74
North-West Frontier     74
Chittagong              68
Yunnan                  67
Dhawalagiri             59
Sikkim                  56
Lumbini                 53
Northern Areas          53
Rapti                   43
Seti                    41
Mizoram                 38
Guizhou                 36
Naraya

In [9]:
study_events = events[
    events["div_name"].isin(["Uttaranchal", "Himachal Pradesh"])
].copy()

# Use the modern state name in our project
study_events["state"] = study_events["div_name"].replace({
    "Uttaranchal": "Uttarakhand"
})

print("Total study events:", len(study_events))
print("\nEvents by state:")
print(study_events["state"].value_counts())

Total study events: 429

Events by state:
state
Uttarakhand         221
Himachal Pradesh    208
Name: count, dtype: int64


In [10]:
print(study_events["ls_trig"].value_counts(dropna=False))

ls_trig
downpour             221
rain                  94
continuous_rain       52
unknown               31
construction          11
monsoon               10
snowfall_snowmelt      5
mining                 4
other                  1
Name: count, dtype: int64


In [11]:
rain_triggers = [
    "downpour",
    "rain",
    "continuous_rain",
    "monsoon"
]

rain_events = study_events[
    study_events["ls_trig"].isin(rain_triggers)
].copy()

print("Rainfall-related events:", len(rain_events))
print("\nBy state:")
print(rain_events["state"].value_counts())

print("\nBy trigger:")
print(rain_events["ls_trig"].value_counts())

Rainfall-related events: 377

By state:
state
Uttarakhand         204
Himachal Pradesh    173
Name: count, dtype: int64

By trigger:
ls_trig
downpour           221
rain                94
continuous_rain     52
monsoon             10
Name: count, dtype: int64


In [12]:
# Check the rainfall-event dates before extracting rainfall

print("Number of rainfall-related events:", len(rain_events))

print("\nDate range:")
print("Earliest:", rain_events["ev_date"].min())
print("Latest:", rain_events["ev_date"].max())

print("\nFirst 10 events:")
display(
    rain_events[
        ["ev_date", "latitude", "longitude", "state", "ls_trig"]
    ].sort_values("ev_date").head(10)
)

Number of rainfall-related events: 377

Date range:
Earliest: 2007-07-30 00:00:00
Latest: 2018-10-09 00:00:00

First 10 events:


,ev_date,latitude,longitude,state,ls_trig
1635,2007-07-30,30.7330,78.4390,Uttarakhand,downpour
1554,2007-07-30,30.4000,79.3490,Uttarakhand,downpour
1031,2007-07-30,30.5570,79.5600,Uttarakhand,downpour
1564,2007-08-11,30.0702,78.3027,Uttarakhand,rain
1894,2007-08-12,32.7334,77.5914,Himachal Pradesh,rain
1612,2007-08-12,32.0985,76.2665,Himachal Pradesh,rain
439,2007-09-06,29.5833,80.2167,Uttarakhand,rain
913,2008-06-03,31.2077,77.1930,Himachal Pradesh,rain
2272,2008-06-18,32.3712,77.2466,Himachal Pradesh,rain
1143,2008-07-18,29.3300,80.1000,Uttarakhand,rain


In [14]:
import ee

print(ee.__version__)

1.7.46


In [18]:
import ee

ee.Authenticate()


Successfully saved authorization token.


In [19]:
import ee

ee.Initialize(project="integral-server-484809-p2")

print("Earth Engine connected successfully!")


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine connected successfully!


In [20]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

print("CHIRPS images:", chirps.size().getInfo())

CHIRPS images: 16679


In [21]:
# Take the first rainfall-related event
event = rain_events.iloc[0]

lat = event["latitude"]
lon = event["longitude"]
event_date = event["ev_date"]

print("Location:", lat, lon)
print("Event date:", event_date)

Location: 30.7666 78.7963
Event date: 2011-06-29 00:00:00


In [22]:
from datetime import timedelta

# Event information
event = rain_events.iloc[0]

lat = float(event["latitude"])
lon = float(event["longitude"])
event_date = event["ev_date"]

# Previous day
previous_date = event_date - timedelta(days=1)

# Create point
point = ee.Geometry.Point([lon, lat])

# Get CHIRPS rainfall for the previous day
rainfall_image = chirps.filterDate(
    previous_date.strftime("%Y-%m-%d"),
    event_date.strftime("%Y-%m-%d")
).first()

# Extract rainfall at event location
rainfall_value = rainfall_image.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=5000
).get("precipitation").getInfo()

print("Event date:", event_date.date())
print("Rainfall on previous day:", rainfall_value, "mm")


Event date: 2011-06-29
Rainfall on previous day: 0 mm


In [23]:
# 3-day antecedent rainfall
start_date = event_date - timedelta(days=3)

rainfall_3d = chirps.filterDate(
    start_date.strftime("%Y-%m-%d"),
    event_date.strftime("%Y-%m-%d")
).sum()

rainfall_3d_value = rainfall_3d.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=5000
).get("precipitation").getInfo()

print("Event date:", event_date.date())
print("3-day antecedent rainfall:", rainfall_3d_value, "mm")

Event date: 2011-06-29
3-day antecedent rainfall: 0 mm


In [24]:
# 15-day antecedent rainfall

start_date = event_date - timedelta(days=15)

rainfall_15d = chirps.filterDate(
    start_date.strftime("%Y-%m-%d"),
    event_date.strftime("%Y-%m-%d")
).sum()

rainfall_15d_value = rainfall_15d.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=5000
).get("precipitation").getInfo()

print("Event date:", event_date.date())
print("15-day antecedent rainfall:", rainfall_15d_value, "mm")

Event date: 2011-06-29
15-day antecedent rainfall: 351.08251953125 mm


In [25]:
# 7-day antecedent rainfall

start_date = event_date - timedelta(days=7)

rainfall_7d = chirps.filterDate(
    start_date.strftime("%Y-%m-%d"),
    event_date.strftime("%Y-%m-%d")
).sum()

rainfall_7d_value = rainfall_7d.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=5000
).get("precipitation").getInfo()

print("Event date:", event_date.date())
print("7-day antecedent rainfall:", rainfall_7d_value, "mm")

Event date: 2011-06-29
7-day antecedent rainfall: 351.08251953125 mm


In [26]:
# 30-day antecedent rainfall

start_date = event_date - timedelta(days=30)

rainfall_30d = chirps.filterDate(
    start_date.strftime("%Y-%m-%d"),
    event_date.strftime("%Y-%m-%d")
).sum()

rainfall_30d_value = rainfall_30d.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=5000
).get("precipitation").getInfo()

print("Event date:", event_date.date())
print("30-day antecedent rainfall:", rainfall_30d_value, "mm")

Event date: 2011-06-29
30-day antecedent rainfall: 351.08251953125 mm


In [27]:
# Prepare rainfall events for Earth Engine extraction

event_records = []

for _, row in rain_events.iterrows():
    event_records.append({
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "event_date": row["ev_date"].strftime("%Y-%m-%d"),
        "state": row["state"],
        "trigger": row["ls_trig"]
    })

print("Events prepared:", len(event_records))
print(event_records[:2])

Events prepared: 377
[{'latitude': 30.7666, 'longitude': 78.7963, 'event_date': '2011-06-29', 'state': 'Uttarakhand', 'trigger': 'downpour'}, {'latitude': 30.7652, 'longitude': 78.5925, 'event_date': '2011-07-09', 'state': 'Uttarakhand', 'trigger': 'downpour'}]


In [28]:
def get_antecedent_rainfall(lat, lon, event_date):
    point = ee.Geometry.Point([lon, lat])

    end_date = ee.Date(event_date)

    rain_1d = chirps.filterDate(
        end_date.advance(-1, "day"), end_date
    ).sum()

    rain_3d = chirps.filterDate(
        end_date.advance(-3, "day"), end_date
    ).sum()

    rain_7d = chirps.filterDate(
        end_date.advance(-7, "day"), end_date
    ).sum()

    rain_15d = chirps.filterDate(
        end_date.advance(-15, "day"), end_date
    ).sum()

    rain_30d = chirps.filterDate(
        end_date.advance(-30, "day"), end_date
    ).sum()

    combined = ee.Image.cat([
        rain_1d.rename("RAIN_1D"),
        rain_3d.rename("RAIN_3D"),
        rain_7d.rename("RAIN_7D"),
        rain_15d.rename("RAIN_15D"),
        rain_30d.rename("RAIN_30D")
    ])

    values = combined.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000
    )

    return values

In [29]:
test_event = event_records[0]

test_values = get_antecedent_rainfall(
    test_event["latitude"],
    test_event["longitude"],
    test_event["event_date"]
).getInfo()

print(test_values)


{'RAIN_15D': 351.08251953125, 'RAIN_1D': 0, 'RAIN_30D': 351.08251953125, 'RAIN_3D': 0, 'RAIN_7D': 351.08251953125}


In [30]:
rainfall_results = []

for i, event in enumerate(event_records):

    values = get_antecedent_rainfall(
        event["latitude"],
        event["longitude"],
        event["event_date"]
    ).getInfo()

    rainfall_results.append({
        "latitude": event["latitude"],
        "longitude": event["longitude"],
        "event_date": event["event_date"],
        "state": event["state"],
        "trigger": event["trigger"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D")
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(event_records)} events")

print("Extraction complete!")

Processed 25/377 events
Processed 50/377 events
Processed 75/377 events
Processed 100/377 events
Processed 125/377 events
Processed 150/377 events
Processed 175/377 events
Processed 200/377 events
Processed 225/377 events
Processed 250/377 events
Processed 275/377 events
Processed 300/377 events
Processed 325/377 events
Processed 350/377 events
Processed 375/377 events
Extraction complete!


In [31]:
rainfall_df = pd.DataFrame(rainfall_results)

rainfall_df.to_csv(
    "../data/himalayan_event_rainfall.csv",
    index=False
)

print("Saved:", rainfall_df.shape)

Saved: (377, 10)


In [2]:
import pandas as pd
rainfall_df = pd.read_csv("../data/himalayan_event_rainfall.csv")

print("Shape:", rainfall_df.shape)
print("\nColumns:")
print(rainfall_df.columns.tolist())

print("\nMissing values:")
print(rainfall_df.isnull().sum())

Shape: (377, 10)

Columns:
['latitude', 'longitude', 'event_date', 'state', 'trigger', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D']

Missing values:
latitude      0
longitude     0
event_date    0
state         0
trigger       0
RAIN_1D       0
RAIN_3D       0
RAIN_7D       0
RAIN_15D      0
RAIN_30D      0
dtype: int64


In [3]:
rainfall_df[
    ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
].describe()

,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D
count,377.000000,377.000000,377.000000,377.000000,377.000000
mean,20.198336,44.716313,95.451626,191.113539,342.786355
std,33.385564,50.532390,87.765039,126.459480,224.398655
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,31.598431,89.277454,156.150055
50%,0.000000,32.284706,73.037144,177.775311,324.715841
75%,30.766394,68.054703,139.426019,271.257778,470.084414
max,221.838257,396.035126,396.035126,649.236227,1328.206873


In [4]:
state_rain = rainfall_df.groupby("state")[
    ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
].agg(["mean", "median"])

state_rain

RAIN_1D           RAIN_3D                RAIN_7D  \
                       mean median       mean     median        mean   
state                                                                  
Himachal Pradesh  18.509474    0.0  39.915555  21.853811   77.712895   
Uttarakhand       21.630557    0.0  48.787544  37.689967  110.494764   

                               RAIN_15D               RAIN_30D              
                     median        mean     median        mean      median  
state                                                                       
Himachal Pradesh  48.804603  168.611723  136.43013  306.823646  269.442939  
Uttarakhand       88.089238  210.195961  214.19686  373.284143  365.279606

In [5]:
static_df = pd.read_csv("../data/himalayan_landslide_v1_dataset.csv")

print("Shape:", static_df.shape)
print("\nColumns:")
print(static_df.columns.tolist())

Shape: (23018, 7)

Columns:
['LATITUDE', 'LONGITUDE', 'ELEVATION', 'SLOPE', 'RAINFALL', 'LANDCOVER_CODE', 'LANDSLIDE']


In [6]:
print("Static latitude range:",
      static_df["LATITUDE"].min(),
      "to",
      static_df["LATITUDE"].max())

print("Static longitude range:",
      static_df["LONGITUDE"].min(),
      "to",
      static_df["LONGITUDE"].max())

print("\nEvent latitude range:",
      rainfall_df["latitude"].min(),
      "to",
      rainfall_df["latitude"].max())

print("Event longitude range:",
      rainfall_df["longitude"].min(),
      "to",
      rainfall_df["longitude"].max())

Static latitude range: 28.727638520868283 to 33.16637488977268
Static longitude range: 75.60910965105909 to 81.0104078485408

Event latitude range: 28.8665 to 32.76390852
Event longitude range: 75.9568 to 80.8576


In [7]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

# Event coordinates
event_coords = rainfall_df[["latitude", "longitude"]].to_numpy()

# Static dataset coordinates
static_coords = static_df[["LATITUDE", "LONGITUDE"]].to_numpy()

# Find nearest static point for each event
nn = NearestNeighbors(n_neighbors=1)
nn.fit(static_coords)

distances, indices = nn.kneighbors(event_coords)

# Convert approximate degree distance to km
nearest_distance_km = distances[:, 0] * 111

print("Nearest-point distance (km):")
print("Minimum:", nearest_distance_km.min())
print("Median :", np.median(nearest_distance_km))
print("Maximum:", nearest_distance_km.max())

Nearest-point distance (km):
Minimum: 0.011100000000368482
Median : 0.789232988062171
Maximum: 4.727530634386611


In [8]:
# Get the nearest static record for each historical event

nearest_static = static_df.iloc[indices[:, 0]].reset_index(drop=True)

event_dataset = rainfall_df.copy()

event_dataset["ELEVATION"] = nearest_static["ELEVATION"].values
event_dataset["SLOPE"] = nearest_static["SLOPE"].values
event_dataset["LANDCOVER_CODE"] = nearest_static["LANDCOVER_CODE"].values

print("Shape:", event_dataset.shape)

display(event_dataset.head())

Shape: (377, 13)


,latitude,longitude,event_date,state,trigger,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D,ELEVATION,SLOPE,LANDCOVER_CODE
0,30.7666,78.7963,2011-06-29,Uttarakhand,downpour,0.000000,0.000000,351.082520,351.082520,351.082520,4471,24.592609,100
1,30.7652,78.5925,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,89.277454,165.996467,1739,41.119884,30
2,30.7331,79.0685,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,0.000000,394.746674,3595,24.119230,100
3,31.0188,78.4129,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,156.171574,372.439198,3533,16.838165,30
4,29.7334,79.6367,2011-07-21,Uttarakhand,rain,49.256058,65.674744,65.674744,152.320995,365.862450,1330,15.842776,30


In [9]:
event_dataset["event_date"] = pd.to_datetime(event_dataset["event_date"])

print("Earliest event:", event_dataset["event_date"].min())
print("Latest event:", event_dataset["event_date"].max())

print("\nEvents by year:")
print(event_dataset["event_date"].dt.year.value_counts().sort_index())

Earliest event: 2007-07-30 00:00:00
Latest event: 2018-10-09 00:00:00

Events by year:
event_date
2007     7
2008     9
2009     8
2010    88
2011    49
2012    15
2013    82
2014     7
2015    39
2016    24
2017    23
2018    26
Name: count, dtype: int64


In [10]:
event_dates = (
    event_dataset["event_date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

date_gaps = event_dates.diff().dt.days.dropna()

print("Unique event dates:", len(event_dates))

print("\nDays between consecutive event dates:")
print(date_gaps.describe())

print("\nNumber of gaps <= 3 days:", (date_gaps <= 3).sum())
print("Number of gaps <= 7 days:", (date_gaps <= 7).sum())

Unique event dates: 212

Days between consecutive event dates:
count    211.000000
mean      19.379147
std       51.899063
min        1.000000
25%        1.000000
50%        3.000000
75%        8.000000
max      323.000000
Name: event_date, dtype: float64

Number of gaps <= 3 days: 115
Number of gaps <= 7 days: 154


In [11]:
print("Latitude range:",
      event_dataset["latitude"].min(),
      "to",
      event_dataset["latitude"].max())

print("Longitude range:",
      event_dataset["longitude"].min(),
      "to",
      event_dataset["longitude"].max())

print("\nEvents by state:")
print(event_dataset["state"].value_counts())

Latitude range: 28.8665 to 32.76390852
Longitude range: 75.9568 to 80.8576

Events by state:
state
Uttarakhand         204
Himachal Pradesh    173
Name: count, dtype: int64


In [12]:
coords = event_dataset[["latitude", "longitude"]]

print("Total events:", len(coords))
print("Exact duplicate coordinates:", coords.duplicated().sum())

print("\nUnique coordinates:", len(coords.drop_duplicates()))

Total events: 377
Exact duplicate coordinates: 3

Unique coordinates: 374


In [13]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

coords = event_dataset[["latitude", "longitude"]].to_numpy()

nn = NearestNeighbors(n_neighbors=2)
nn.fit(coords)

distances, indices_near = nn.kneighbors(coords)

# Nearest neighbor excluding the point itself
nearest_km = distances[:, 1] * 111

print("Nearest-neighbor distance:")
print("Minimum:", nearest_km.min(), "km")
print("Median :", np.median(nearest_km), "km")
print("Maximum:", nearest_km.max(), "km")

print("\nEvents with another event within:")
print("1 km :", (nearest_km <= 1).sum())
print("5 km :", (nearest_km <= 5).sum())
print("10 km:", (nearest_km <= 10).sum())

Nearest-neighbor distance:
Minimum: 0.0 km
Median : 1.98841869333445 km
Maximum: 47.72818884368994 km

Events with another event within:
1 km : 140
5 km : 270
10 km: 315


In [14]:
start_date = event_dataset["event_date"].min()
end_date = event_dataset["event_date"].max()

all_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

print("Timeline start:", start_date.date())
print("Timeline end:", end_date.date())
print("Number of days:", len(all_dates))

Timeline start: 2007-07-30
Timeline end: 2018-10-09
Number of days: 4090


In [15]:
prediction_dates = all_dates[
    all_dates <= end_date - pd.Timedelta(days=3)
]

print("Valid prediction dates:", len(prediction_dates))
print("First:", prediction_dates.min().date())
print("Last :", prediction_dates.max().date())

Valid prediction dates: 4087
First: 2007-07-30
Last : 2018-10-06


In [16]:
feature_start_date = start_date + pd.Timedelta(days=30)

prediction_dates = all_dates[
    (all_dates >= feature_start_date) &
    (all_dates <= end_date - pd.Timedelta(days=3))
]

print("Feature start:", feature_start_date.date())
print("Last prediction date:", prediction_dates.max().date())
print("Valid prediction dates:", len(prediction_dates))

Feature start: 2007-08-29
Last prediction date: 2018-10-06
Valid prediction dates: 4057


In [17]:
event_date_set = set(
    event_dataset["event_date"].dt.normalize()
)

print("Unique rainfall-event dates:", len(event_date_set))

print("Example dates:")
print(list(sorted(event_date_set))[:10])

Unique rainfall-event dates: 212
Example dates:
[Timestamp('2007-07-30 00:00:00'), Timestamp('2007-08-11 00:00:00'), Timestamp('2007-08-12 00:00:00'), Timestamp('2007-09-06 00:00:00'), Timestamp('2008-06-03 00:00:00'), Timestamp('2008-06-18 00:00:00'), Timestamp('2008-07-18 00:00:00'), Timestamp('2008-08-08 00:00:00'), Timestamp('2008-08-14 00:00:00'), Timestamp('2008-08-17 00:00:00')]


In [18]:
target_rows = []

for date in prediction_dates:
    future_dates = {
        date + pd.Timedelta(days=1),
        date + pd.Timedelta(days=2),
        date + pd.Timedelta(days=3)
    }

    target = int(
        any(d in event_date_set for d in future_dates)
    )

    target_rows.append({
        "reference_date": date,
        "LANDSLIDE_NEXT_3D": target
    })

target_df = pd.DataFrame(target_rows)

print("Shape:", target_df.shape)
print("\nTarget distribution:")
print(target_df["LANDSLIDE_NEXT_3D"].value_counts())

print("\nPercentage:")
print(
    target_df["LANDSLIDE_NEXT_3D"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Shape: (4057, 2)

Target distribution:
LANDSLIDE_NEXT_3D
0    3589
1     468
Name: count, dtype: int64

Percentage:
LANDSLIDE_NEXT_3D
0    88.46
1    11.54
Name: proportion, dtype: float64


In [19]:
event_locations = (
    event_dataset[
        ["latitude", "longitude", "state"]
    ]
    .drop_duplicates(
        subset=["latitude", "longitude"]
    )
    .reset_index(drop=True)
)

print("Unique event locations:", len(event_locations))
print("\nBy state:")
print(event_locations["state"].value_counts())

display(event_locations.head())

Unique event locations: 374

By state:
state
Uttarakhand         202
Himachal Pradesh    172
Name: count, dtype: int64


,latitude,longitude,state
0,30.7666,78.7963,Uttarakhand
1,30.7652,78.5925,Uttarakhand
2,30.7331,79.0685,Uttarakhand
3,31.0188,78.4129,Uttarakhand
4,29.7334,79.6367,Uttarakhand


In [20]:
event_months = (
    event_dataset["event_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print("Number of active months:", len(event_months))
print("\nEvents by month:")
print(event_months)

Number of active months: 55

Events by month:
event_date
2007-07     3
2007-08     3
2007-09     1
2008-06     2
2008-07     1
2008-08     3
2008-09     3
2009-08     2
2009-09     6
2010-02     1
2010-06     1
2010-07    22
2010-08    21
2010-09    40
2010-10     3
2011-02     2
2011-03     1
2011-05     2
2011-06    13
2011-07    18
2011-08    12
2011-12     1
2012-02     1
2012-07     2
2012-08     9
2012-09     3
2013-06    25
2013-07    43
2013-08    12
2013-10     2
2014-07     4
2014-08     2
2014-09     1
2015-03     1
2015-04     3
2015-06     2
2015-07    21
2015-08    12
2016-05     2
2016-06     3
2016-07     9
2016-08    10
2017-01     1
2017-02     1
2017-04     5
2017-05     1
2017-06     5
2017-07     8
2017-08     1
2017-09     1
2018-05     1
2018-06     1
2018-08    12
2018-09    11
2018-10     1
Freq: M, Name: count, dtype: int64


In [21]:
coords = event_locations[["latitude", "longitude"]].to_numpy()

nn = NearestNeighbors(n_neighbors=2)
nn.fit(coords)

distances, _ = nn.kneighbors(coords)

nearest_km = distances[:, 1] * 111

print("Unique event locations:", len(event_locations))

print("\nNearest-location distance:")
print("Minimum:", nearest_km.min(), "km")
print("Median :", np.median(nearest_km), "km")

print("\nLocations with nearest neighbor farther than:")
print("1 km :", (nearest_km > 1).sum())
print("5 km :", (nearest_km > 5).sum())
print("10 km:", (nearest_km > 10).sum())

Unique event locations: 374

Nearest-location distance:
Minimum: 0.02219999999994826 km
Median : 2.1141567318608643 km

Locations with nearest neighbor farther than:
1 km : 240
5 km : 109
10 km: 63


In [22]:
for state, group in event_locations.groupby("state"):
    print(f"\n{state}")
    print("Latitude :", group["latitude"].min(), "to", group["latitude"].max())
    print("Longitude:", group["longitude"].min(), "to", group["longitude"].max())
    print("Locations:", len(group))


Himachal Pradesh
Latitude : 30.5589 to 32.76390852
Longitude: 75.9568 to 78.6167
Locations: 172

Uttarakhand
Latitude : 28.8665 to 31.0358
Longitude: 77.8644 to 80.8576
Locations: 202


In [24]:
import ee

ee.Authenticate()

True

In [25]:
ee.Initialize(project="integral-server-484809-p2")

print("Earth Engine connected successfully!")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine connected successfully!


In [26]:
states = ee.FeatureCollection("FAO/GAUL/2015/level1")

study_region = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.inList(
            "ADM1_NAME",
            ["Himachal Pradesh", "Uttarakhand"]
        )
    )
)

print("Number of state features:", study_region.size().getInfo())

C:\Users\imman\AppData\Roaming\Python\Python311\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


Number of state features: 2


In [27]:
candidate_points = ee.FeatureCollection.randomPoints(
    region=study_region.geometry(),
    points=1000,
    seed=42,
    maxError=100
)

print(
    "Candidate points:",
    candidate_points.size().getInfo()
)

Candidate points: 1000


In [28]:
event_points = ee.FeatureCollection([
    ee.Feature(
        ee.Geometry.Point([row["longitude"], row["latitude"]])
    )
    for _, row in event_locations.iterrows()
])

print("Event locations:", event_points.size().getInfo())

Event locations: 374


In [29]:
event_buffer = event_points.geometry().buffer(1000)

background_test = candidate_points.filter(
    ee.Filter.disjoint(".geo", event_buffer)
)

print(
    "Background candidates outside 1 km:",
    background_test.size().getInfo()
)

Background candidates outside 1 km: 988


In [30]:
candidate_with_state = ee.FeatureCollection.randomPoints(
    region=study_region.geometry(),
    points=1000,
    seed=42,
    maxError=100
)

print("Candidate count:", candidate_with_state.size().getInfo())

Candidate count: 1000


In [31]:
hp_region = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.eq("ADM1_NAME", "Himachal Pradesh")
    )
)

uk_region = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.eq("ADM1_NAME", "Uttarakhand")
    )
)

hp_candidates = ee.FeatureCollection.randomPoints(
    region=hp_region.geometry(),
    points=500,
    seed=42,
    maxError=100
)

uk_candidates = ee.FeatureCollection.randomPoints(
    region=uk_region.geometry(),
    points=500,
    seed=43,
    maxError=100
)

print("Himachal candidates:", hp_candidates.size().getInfo())
print("Uttarakhand candidates:", uk_candidates.size().getInfo())

Himachal candidates: 500
Uttarakhand candidates: 500


In [32]:
event_buffer = event_points.geometry().buffer(1000)

hp_background_candidates = hp_candidates.filter(
    ee.Filter.disjoint(".geo", event_buffer)
)

uk_background_candidates = uk_candidates.filter(
    ee.Filter.disjoint(".geo", event_buffer)
)

print(
    "Himachal background candidates:",
    hp_background_candidates.size().getInfo()
)

print(
    "Uttarakhand background candidates:",
    uk_background_candidates.size().getInfo()
)

Himachal background candidates: 493
Uttarakhand background candidates: 495


In [33]:
hp_background = hp_background_candidates.limit(
    172, "system:index", False
)

uk_background = uk_background_candidates.limit(
    202, "system:index", False
)

print("HP background:", hp_background.size().getInfo())
print("UK background:", uk_background.size().getInfo())

HP background: 172
UK background: 202


In [34]:
background_points = hp_background.merge(uk_background)

background_coords = background_points.getInfo()

background_locations = []

for feature in background_coords["features"]:
    lon, lat = feature["geometry"]["coordinates"]

    background_locations.append({
        "latitude": lat,
        "longitude": lon
    })

background_locations = pd.DataFrame(background_locations)

print("Background locations:", len(background_locations))
display(background_locations.head())

Background locations: 374


,latitude,longitude
0,31.803484,76.244918
1,31.451544,78.762136
2,31.589047,78.027487
3,32.934202,76.517776
4,32.636886,78.325482


In [35]:
hp_coords = hp_background.getInfo()["features"]
uk_coords = uk_background.getInfo()["features"]

hp_set = {
    (
        f["geometry"]["coordinates"][0],
        f["geometry"]["coordinates"][1]
    )
    for f in hp_coords
}

uk_set = {
    (
        f["geometry"]["coordinates"][0],
        f["geometry"]["coordinates"][1]
    )
    for f in uk_coords
}

background_locations["state"] = background_locations.apply(
    lambda row: (
        "Himachal Pradesh"
        if (row["longitude"], row["latitude"]) in hp_set
        else "Uttarakhand"
    ),
    axis=1
)

print(background_locations["state"].value_counts())


state
Uttarakhand         202
Himachal Pradesh    172
Name: count, dtype: int64


In [36]:
positive_samples = event_dataset.copy()

positive_samples["reference_date"] = (
    positive_samples["event_date"]
    - pd.Timedelta(days=3)
)

positive_samples["LANDSLIDE_NEXT_3D"] = 1

print("Positive samples:", len(positive_samples))

display(
    positive_samples[
        [
            "latitude",
            "longitude",
            "event_date",
            "reference_date",
            "LANDSLIDE_NEXT_3D"
        ]
    ].head()
)

Positive samples: 377


,latitude,longitude,event_date,reference_date,LANDSLIDE_NEXT_3D
0,30.7666,78.7963,2011-06-29,2011-06-26,1
1,30.7652,78.5925,2011-07-09,2011-07-06,1
2,30.7331,79.0685,2011-07-09,2011-07-06,1
3,31.0188,78.4129,2011-07-09,2011-07-06,1
4,29.7334,79.6367,2011-07-21,2011-07-18,1


In [37]:
import numpy as np

np.random.seed(42)

negative_dates = np.random.choice(
    prediction_dates,
    size=377,
    replace=False
)

negative_dates = pd.to_datetime(negative_dates)

print("Negative dates:", len(negative_dates))
print("First 10:")
print(negative_dates[:10])

Negative dates: 377
First 10:
DatetimeIndex(['2011-11-15', '2014-04-09', '2018-10-04', '2009-09-13',
               '2013-01-21', '2009-02-13', '2008-02-11', '2016-12-02',
               '2014-04-07', '2017-09-05'],
              dtype='datetime64[ns]', freq=None)


In [39]:
print(type(background_locations))
print(background_locations.shape)
print(background_locations.head())

<class 'pandas.core.frame.DataFrame'>
(374, 3)
    latitude  longitude             state
0  31.803484  76.244918  Himachal Pradesh
1  31.451544  78.762136  Himachal Pradesh
2  31.589047  78.027487  Himachal Pradesh
3  32.934202  76.517776  Himachal Pradesh
4  32.636886  78.325482  Himachal Pradesh


In [40]:
negative_candidates = background_locations.sample(
    n=377,
    random_state=42,
    replace=True
).reset_index(drop=True)

negative_candidates["reference_date"] = negative_dates

print("Candidate negative samples:", len(negative_candidates))

display(negative_candidates.head())

Candidate negative samples: 377


,latitude,longitude,state,reference_date
0,32.066038,75.950161,Himachal Pradesh,2011-11-15
1,30.043504,78.474039,Uttarakhand,2014-04-09
2,30.032687,79.206576,Uttarakhand,2018-10-04
3,32.466893,76.097220,Himachal Pradesh,2009-09-13
4,32.357668,78.292560,Himachal Pradesh,2013-01-21


In [41]:
location_event_dates = (
    event_dataset
    .groupby(["latitude", "longitude"])["event_date"]
    .apply(lambda x: sorted(x.dt.normalize().unique()))
    .to_dict()
)

print("Locations with event history:", len(location_event_dates))

# Show one example
first_location = next(iter(location_event_dates))

print("\nExample location:", first_location)
print("Event dates:", location_event_dates[first_location])

Locations with event history: 374

Example location: (28.8665, 79.5682)
Event dates: [Timestamp('2011-08-12 00:00:00')]


In [42]:
np.random.seed(42)

safe_negative_samples = []

for (lat, lon), event_dates in location_event_dates.items():

    event_dates = set(pd.to_datetime(event_dates))

    # Candidate reference dates for this location
    candidates = prediction_dates.to_list()

    # Keep dates whose next 3 days contain no recorded event
    safe_dates = []

    for date in candidates:
        future_dates = {
            date + pd.Timedelta(days=1),
            date + pd.Timedelta(days=2),
            date + pd.Timedelta(days=3)
        }

        if not any(d in event_dates for d in future_dates):
            safe_dates.append(date)

    # Randomly choose up to 1 negative date for this location
    if safe_dates:
        chosen_date = np.random.choice(safe_dates)

        safe_negative_samples.append({
            "latitude": lat,
            "longitude": lon,
            "reference_date": pd.Timestamp(chosen_date)
        })

safe_negative_df = pd.DataFrame(safe_negative_samples)

print("Same-location negative samples:", len(safe_negative_df))
print("\nUnique locations represented:",
      safe_negative_df[["latitude", "longitude"]].drop_duplicates().shape[0])

display(safe_negative_df.head())

Same-location negative samples: 374

Unique locations represented: 374


,latitude,longitude,reference_date
0,28.866500,79.568200,2016-05-10
1,29.158500,79.146400,2017-04-08
2,29.295586,79.540431,2010-01-05
3,29.330000,80.100000,2011-03-18
4,29.337636,79.483545,2010-10-05


In [43]:
negative_check = []

for _, row in safe_negative_df.iterrows():

    key = (row["latitude"], row["longitude"])

    event_dates = set(location_event_dates[key])

    future_dates = {
        row["reference_date"] + pd.Timedelta(days=1),
        row["reference_date"] + pd.Timedelta(days=2),
        row["reference_date"] + pd.Timedelta(days=3)
    }

    has_event = any(d in event_dates for d in future_dates)

    negative_check.append(has_event)

print("Unexpected positive cases:", sum(negative_check))
print("Total checked:", len(negative_check))

Unexpected positive cases: 0
Total checked: 374


In [45]:
def get_antecedent_rainfall(lat, lon, event_date):
    point = ee.Geometry.Point([lon, lat])
    end_date = ee.Date(event_date)

    rain_1d = chirps.filterDate(
        end_date.advance(-1, "day"), end_date
    ).sum()

    rain_3d = chirps.filterDate(
        end_date.advance(-3, "day"), end_date
    ).sum()

    rain_7d = chirps.filterDate(
        end_date.advance(-7, "day"), end_date
    ).sum()

    rain_15d = chirps.filterDate(
        end_date.advance(-15, "day"), end_date
    ).sum()

    rain_30d = chirps.filterDate(
        end_date.advance(-30, "day"), end_date
    ).sum()

    combined = ee.Image.cat([
        rain_1d.rename("RAIN_1D"),
        rain_3d.rename("RAIN_3D"),
        rain_7d.rename("RAIN_7D"),
        rain_15d.rename("RAIN_15D"),
        rain_30d.rename("RAIN_30D")
    ])

    return combined.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000
    )

In [46]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

In [47]:
test_negative = safe_negative_df.iloc[0]

test_values = get_antecedent_rainfall(
    float(test_negative["latitude"]),
    float(test_negative["longitude"]),
    test_negative["reference_date"].strftime("%Y-%m-%d")
).getInfo()

print(test_values)

{'RAIN_15D': 20.463307857513428, 'RAIN_1D': 0, 'RAIN_30D': 20.463307857513428, 'RAIN_3D': 0, 'RAIN_7D': 20.463307857513428}


In [48]:
negative_records = []

for i, row in safe_negative_df.iterrows():

    values = get_antecedent_rainfall(
        float(row["latitude"]),
        float(row["longitude"]),
        row["reference_date"].strftime("%Y-%m-%d")
    ).getInfo()

    negative_records.append({
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "reference_date": row["reference_date"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D"),
        "LANDSLIDE_NEXT_3D": 0
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/374")

negative_temporal = pd.DataFrame(negative_records)

print("Extraction complete!")
print("Shape:", negative_temporal.shape)

Processed 25/374
Processed 50/374
Processed 75/374
Processed 100/374
Processed 125/374
Processed 150/374
Processed 175/374
Processed 200/374
Processed 225/374
Processed 250/374
Processed 275/374
Processed 300/374
Processed 325/374
Processed 350/374
Extraction complete!
Shape: (374, 9)


In [49]:
negative_temporal.to_csv(
    "../data/negative_temporal_samples.csv",
    index=False
)

print("Saved:", negative_temporal.shape)


Saved: (374, 9)


In [50]:
print(negative_temporal.isnull().sum())

print("\nTarget:")
print(negative_temporal["LANDSLIDE_NEXT_3D"].value_counts())

latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
LANDSLIDE_NEXT_3D    0
dtype: int64

Target:
LANDSLIDE_NEXT_3D
0    374
Name: count, dtype: int64


In [51]:
positive_temporal = positive_samples[
    [
        "latitude",
        "longitude",
        "reference_date",
        "RAIN_1D",
        "RAIN_3D",
        "RAIN_7D",
        "RAIN_15D",
        "RAIN_30D",
        "ELEVATION",
        "SLOPE",
        "LANDCOVER_CODE"
    ]
].copy()

positive_temporal["LANDSLIDE_NEXT_3D"] = 1

print("Positive samples:", positive_temporal.shape)

display(positive_temporal.head())

Positive samples: (377, 12)


,latitude,longitude,reference_date,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D,ELEVATION,SLOPE,LANDCOVER_CODE,LANDSLIDE_NEXT_3D
0,30.7666,78.7963,2011-06-26,0.000000,0.000000,351.082520,351.082520,351.082520,4471,24.592609,100,1
1,30.7652,78.5925,2011-07-06,0.000000,0.000000,0.000000,89.277454,165.996467,1739,41.119884,30,1
2,30.7331,79.0685,2011-07-06,0.000000,0.000000,0.000000,0.000000,394.746674,3595,24.119230,100,1
3,31.0188,78.4129,2011-07-06,0.000000,0.000000,0.000000,156.171574,372.439198,3533,16.838165,30,1
4,29.7334,79.6367,2011-07-18,49.256058,65.674744,65.674744,152.320995,365.862450,1330,15.842776,30,1


In [52]:
test_positive = positive_samples.iloc[0]

corrected_values = get_antecedent_rainfall(
    float(test_positive["latitude"]),
    float(test_positive["longitude"]),
    test_positive["reference_date"].strftime("%Y-%m-%d")
).getInfo()

print("Reference date:", test_positive["reference_date"].date())
print("Event date:", test_positive["event_date"].date())
print(corrected_values)

Reference date: 2011-06-26
Event date: 2011-06-29
{'RAIN_15D': 351.08251953125, 'RAIN_1D': 0, 'RAIN_30D': 351.08251953125, 'RAIN_3D': 175.541259765625, 'RAIN_7D': 351.08251953125}


In [53]:
positive_records = []

for i, row in positive_samples.iterrows():

    values = get_antecedent_rainfall(
        float(row["latitude"]),
        float(row["longitude"]),
        row["reference_date"].strftime("%Y-%m-%d")
    ).getInfo()

    positive_records.append({
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "reference_date": row["reference_date"],
        "event_date": row["event_date"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D"),
        "ELEVATION": row["ELEVATION"],
        "SLOPE": row["SLOPE"],
        "LANDCOVER_CODE": row["LANDCOVER_CODE"],
        "LANDSLIDE_NEXT_3D": 1
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} / {len(positive_samples)}")

positive_temporal = pd.DataFrame(positive_records)

positive_temporal.to_csv(
    "../data/positive_temporal_samples.csv",
    index=False
)

print("\nSaved:", positive_temporal.shape)
print(positive_temporal.head())

Processed 25 / 377
Processed 50 / 377
Processed 75 / 377
Processed 100 / 377
Processed 125 / 377
Processed 150 / 377
Processed 175 / 377
Processed 200 / 377
Processed 225 / 377
Processed 250 / 377
Processed 275 / 377
Processed 300 / 377
Processed 325 / 377
Processed 350 / 377
Processed 375 / 377

Saved: (377, 13)
   latitude  longitude reference_date event_date  RAIN_1D    RAIN_3D  \
0   30.7666    78.7963     2011-06-26 2011-06-29      0.0  175.54126   
1   30.7652    78.5925     2011-07-06 2011-07-09      0.0    0.00000   
2   30.7331    79.0685     2011-07-06 2011-07-09      0.0    0.00000   
3   31.0188    78.4129     2011-07-06 2011-07-09      0.0    0.00000   
4   29.7334    79.6367     2011-07-18 2011-07-21      0.0    0.00000   

      RAIN_7D    RAIN_15D    RAIN_30D  ELEVATION      SLOPE  LANDCOVER_CODE  \
0  351.082520  351.082520  351.082520       4471  24.592609             100   
1   19.572652  130.109631  172.510793       1739  41.119884              30   
2    0.000000  

In [54]:
print("Shape:", positive_temporal.shape)

print("\nMissing values:")
print(positive_temporal.isnull().sum())

print("\nTarget distribution:")
print(positive_temporal["LANDSLIDE_NEXT_3D"].value_counts())

print("\nDate range:")
print(
    positive_temporal["reference_date"].min(),
    "to",
    positive_temporal["reference_date"].max()
)

print("\nRainfall summary:")
print(
    positive_temporal[
        ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
    ].describe()
)


Shape: (377, 13)

Missing values:
latitude             0
longitude            0
reference_date       0
event_date           0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
ELEVATION            0
SLOPE                0
LANDCOVER_CODE       0
LANDSLIDE_NEXT_3D    0
dtype: int64

Target distribution:
LANDSLIDE_NEXT_3D
1    377
Name: count, dtype: int64

Date range:
2007-07-27 00:00:00 to 2018-10-06 00:00:00

Rainfall summary:
          RAIN_1D     RAIN_3D     RAIN_7D    RAIN_15D     RAIN_30D
count  377.000000  377.000000  377.000000  377.000000   377.000000
mean    10.383832   40.452351   86.154412  185.661282   327.936674
std     21.559562   51.844458   75.328783  131.733761   228.156374
min      0.000000    0.000000    0.000000    0.000000     0.000000
25%      0.000000    0.000000   25.708485   78.803505   139.479708
50%      0.000000   23.469419   70.778240  166.736160   310.892866
75%     10.826213   62.134907  130.

In [55]:
# Load negative samples
negative_temporal = pd.read_csv(
    "../data/negative_temporal_samples.csv",
    parse_dates=["reference_date"]
)

# Get static features from event locations
static_features = (
    event_dataset[
        ["latitude", "longitude", "ELEVATION", "SLOPE", "LANDCOVER_CODE"]
    ]
    .drop_duplicates(subset=["latitude", "longitude"])
)

# Merge static features onto negative samples
negative_temporal = negative_temporal.merge(
    static_features,
    on=["latitude", "longitude"],
    how="left"
)

print("Shape:", negative_temporal.shape)

print("\nMissing values:")
print(negative_temporal.isnull().sum())

print("\nTarget distribution:")
print(negative_temporal["LANDSLIDE_NEXT_3D"].value_counts())

print("\nSample:")
print(negative_temporal.head())

Shape: (374, 12)

Missing values:
latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
LANDSLIDE_NEXT_3D    0
ELEVATION            0
SLOPE                0
LANDCOVER_CODE       0
dtype: int64

Target distribution:
LANDSLIDE_NEXT_3D
0    374
Name: count, dtype: int64

Sample:
    latitude  longitude reference_date  RAIN_1D   RAIN_3D    RAIN_7D  \
0  28.866500  79.568200     2016-05-10      0.0  0.000000  20.463308   
1  29.158500  79.146400     2017-04-08      0.0  0.000000   0.000000   
2  29.295586  79.540431     2010-01-05      0.0  4.916282   4.916282   
3  29.330000  80.100000     2011-03-18      0.0  0.000000   0.000000   
4  29.337636  79.483545     2010-10-05      0.0  0.000000   0.000000   

    RAIN_15D    RAIN_30D  LANDSLIDE_NEXT_3D  ELEVATION      SLOPE  \
0  20.463308   20.463308                  0        206   3.492825   
1   8.249058    8.2490

In [56]:
# Combine positive and negative samples
temporal_dataset = pd.concat(
    [positive_temporal, negative_temporal],
    ignore_index=True
)

# Keep only the features we will use for the first model
temporal_dataset = temporal_dataset[
    [
        "latitude",
        "longitude",
        "reference_date",
        "RAIN_1D",
        "RAIN_3D",
        "RAIN_7D",
        "RAIN_15D",
        "RAIN_30D",
        "ELEVATION",
        "SLOPE",
        "LANDCOVER_CODE",
        "LANDSLIDE_NEXT_3D"
    ]
]

# Shuffle the rows
temporal_dataset = temporal_dataset.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Save
temporal_dataset.to_csv(
    "../data/himalayan_temporal_dataset.csv",
    index=False
)

print("Shape:", temporal_dataset.shape)

print("\nTarget distribution:")
print(temporal_dataset["LANDSLIDE_NEXT_3D"].value_counts())

print("\nMissing values:")
print(temporal_dataset.isnull().sum())

print("\nFirst 5 rows:")
print(temporal_dataset.head())

Shape: (751, 12)

Target distribution:
LANDSLIDE_NEXT_3D
1    377
0    374
Name: count, dtype: int64

Missing values:
latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
ELEVATION            0
SLOPE                0
LANDCOVER_CODE       0
LANDSLIDE_NEXT_3D    0
dtype: int64

First 5 rows:
    latitude  longitude reference_date  RAIN_1D    RAIN_3D     RAIN_7D  \
0  30.404100  79.327600     2010-10-01      0.0   0.000000    0.000000   
1  32.025479  76.166926     2018-08-10      0.0  67.815437  198.701147   
2  29.912100  78.172300     2011-08-09      0.0  24.181955   66.173952   
3  29.158500  79.146400     2014-07-20      0.0  82.999393  141.695946   
4  29.337636  79.483545     2010-09-17      0.0  57.555267  143.888168   

     RAIN_15D    RAIN_30D  ELEVATION      SLOPE  LANDCOVER_CODE  \
0  111.756538  225.169464        991  42.686272              30   

In [57]:
print("Positive reference dates:")
print(
    temporal_dataset[
        temporal_dataset["LANDSLIDE_NEXT_3D"] == 1
    ]["reference_date"].describe()
)

print("\nNegative reference dates:")
print(
    temporal_dataset[
        temporal_dataset["LANDSLIDE_NEXT_3D"] == 0
    ]["reference_date"].describe()
)

print("\nPositive samples by year:")
print(
    temporal_dataset[
        temporal_dataset["LANDSLIDE_NEXT_3D"] == 1
    ]["reference_date"].dt.year.value_counts().sort_index()
)

print("\nNegative samples by year:")
print(
    temporal_dataset[
        temporal_dataset["LANDSLIDE_NEXT_3D"] == 0
    ]["reference_date"].dt.year.value_counts().sort_index()
)

Positive reference dates:
count                              377
mean     2013-04-01 04:04:27.374005248
min                2007-07-27 00:00:00
25%                2010-09-16 00:00:00
50%                2013-06-13 00:00:00
75%                2015-07-21 00:00:00
max                2018-10-06 00:00:00
Name: reference_date, dtype: object

Negative reference dates:
count                              374
mean     2013-05-29 18:44:16.684492032
min                2007-08-30 00:00:00
25%                2010-10-27 06:00:00
50%                2013-04-22 00:00:00
75%                2016-02-13 06:00:00
max                2018-09-09 00:00:00
Name: reference_date, dtype: object

Positive samples by year:
reference_date
2007     7
2008     9
2009     8
2010    88
2011    49
2012    15
2013    82
2014     7
2015    39
2016    24
2017    23
2018    26
Name: count, dtype: int64

Negative samples by year:
reference_date
2007     6
2008    28
2009    32
2010    33
2011    38
2012    36
2013    36
2014    32

In [58]:
year_table = pd.crosstab(
    temporal_dataset["reference_date"].dt.year,
    temporal_dataset["LANDSLIDE_NEXT_3D"]
)

year_table.columns = ["NEGATIVE", "POSITIVE"]

print(year_table)

print("\nPositive/Negative ratio by year:")
print(
    (year_table["POSITIVE"] / year_table["NEGATIVE"])
    .round(2)
)

                NEGATIVE  POSITIVE
reference_date                    
2007                   6         7
2008                  28         9
2009                  32         8
2010                  33        88
2011                  38        49
2012                  36        15
2013                  36        82
2014                  32         7
2015                  36        39
2016                  33        24
2017                  39        23
2018                  25        26

Positive/Negative ratio by year:
reference_date
2007    1.17
2008    0.32
2009    0.25
2010    2.67
2011    1.29
2012    0.42
2013    2.28
2014    0.22
2015    1.08
2016    0.73
2017    0.59
2018    1.04
dtype: float64


In [59]:
positive_dates = (
    temporal_dataset[
        temporal_dataset["LANDSLIDE_NEXT_3D"] == 1
    ]["reference_date"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
)

print("Positive samples:", 377)
print("Unique positive reference dates:", len(positive_dates))

print("\nFirst 20:")
print(positive_dates.head(20).to_list())

print("\nNumber of positive dates by year:")
print(positive_dates.dt.year.value_counts().sort_index())


Positive samples: 377
Unique positive reference dates: 212

First 20:
[Timestamp('2007-07-27 00:00:00'), Timestamp('2007-08-08 00:00:00'), Timestamp('2007-08-09 00:00:00'), Timestamp('2007-09-03 00:00:00'), Timestamp('2008-05-31 00:00:00'), Timestamp('2008-06-15 00:00:00'), Timestamp('2008-07-15 00:00:00'), Timestamp('2008-08-05 00:00:00'), Timestamp('2008-08-11 00:00:00'), Timestamp('2008-08-14 00:00:00'), Timestamp('2008-09-16 00:00:00'), Timestamp('2009-08-05 00:00:00'), Timestamp('2009-08-14 00:00:00'), Timestamp('2009-08-29 00:00:00'), Timestamp('2009-08-31 00:00:00'), Timestamp('2009-09-06 00:00:00'), Timestamp('2009-09-07 00:00:00'), Timestamp('2009-09-09 00:00:00'), Timestamp('2010-02-20 00:00:00'), Timestamp('2010-06-06 00:00:00')]

Number of positive dates by year:
reference_date
2007     4
2008     7
2009     7
2010    38
2011    26
2012     8
2013    35
2014     7
2015    24
2016    20
2017    16
2018    20
Name: count, dtype: int64


In [60]:
event_count_by_date = (
    positive_samples
    .groupby("reference_date")
    .size()
    .sort_values(ascending=False)
)

print("Number of unique positive dates:", len(event_count_by_date))

print("\nEvents per positive date:")
print(event_count_by_date.describe())

print("\nTop dates with most events:")
print(event_count_by_date.head(15))

Number of unique positive dates: 212

Events per positive date:
count    212.000000
mean       1.778302
std        1.806492
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       19.000000
dtype: float64

Top dates with most events:
reference_date
2010-09-16    19
2013-06-13    10
2011-06-26     8
2013-07-04     7
2010-07-04     6
2010-09-17     6
2015-07-23     6
2013-07-21     6
2010-08-17     5
2011-08-04     5
2010-07-22     5
2013-07-22     5
2013-07-11     4
2015-08-12     4
2015-07-21     4
dtype: int64


In [61]:
# All unique event locations
event_locations = (
    event_dataset[
        ["latitude", "longitude", "event_date"]
    ]
    .drop_duplicates()
)

# Normalize dates
event_locations["event_date"] = pd.to_datetime(
    event_locations["event_date"]
).dt.normalize()

positive_dates = positive_samples["reference_date"].dt.normalize().unique()

available_counts = []

for ref_date in positive_dates:

    # Locations having an event during the prediction window
    future_events = event_locations[
        (event_locations["event_date"] > ref_date) &
        (event_locations["event_date"] <= ref_date + pd.Timedelta(days=3))
    ]

    occupied = set(
        zip(
            future_events["latitude"],
            future_events["longitude"]
        )
    )

    available = len(background_locations)

    # Count background locations not among event locations
    blocked = sum(
        (lat, lon) in occupied
        for lat, lon in zip(
            background_locations["latitude"],
            background_locations["longitude"]
        )
    )

    available_counts.append({
        "reference_date": ref_date,
        "available_background": available - blocked
    })

available_df = pd.DataFrame(available_counts)

print(available_df["available_background"].describe())

print("\nMinimum available:", available_df["available_background"].min())
print("Maximum available:", available_df["available_background"].max())

print("\nDates with fewest available locations:")
print(
    available_df
    .sort_values("available_background")
    .head(10)
)

count    212.0
mean     374.0
std        0.0
min      374.0
25%      374.0
50%      374.0
75%      374.0
max      374.0
Name: available_background, dtype: float64

Minimum available: 374
Maximum available: 374

Dates with fewest available locations:
    reference_date  available_background
0       2011-06-26                   374
135     2015-07-29                   374
136     2015-07-16                   374
137     2010-08-11                   374
138     2013-10-15                   374
139     2013-07-28                   374
140     2014-08-15                   374
141     2010-07-03                   374
142     2015-08-06                   374
143     2007-08-08                   374


In [62]:
import pandas as pd
import numpy as np

# Event history by exact location
location_event_dates = (
    event_dataset
    .assign(event_date=pd.to_datetime(event_dataset["event_date"]).dt.normalize())
    .groupby(["latitude", "longitude"])["event_date"]
    .apply(set)
    .to_dict()
)

rng = np.random.default_rng(42)

matched_negative_samples = []

for _, positive_row in positive_samples.iterrows():

    ref_date = pd.Timestamp(positive_row["reference_date"]).normalize()

    # Candidate background locations
    candidates = background_locations[
        ["latitude", "longitude"]
    ].copy()

    # Remove locations that have an event during T+1 to T+3
    valid_locations = []

    for _, loc in candidates.iterrows():

        key = (loc["latitude"], loc["longitude"])
        event_dates = location_event_dates.get(key, set())

        has_future_event = any(
            ref_date < d <= ref_date + pd.Timedelta(days=3)
            for d in event_dates
        )

        if not has_future_event:
            valid_locations.append(
                (loc["latitude"], loc["longitude"])
            )

    # Select one deterministic random negative location
    selected = valid_locations[
        rng.integers(len(valid_locations))
    ]

    matched_negative_samples.append({
        "latitude": selected[0],
        "longitude": selected[1],
        "reference_date": ref_date,
        "LANDSLIDE_NEXT_3D": 0
    })

matched_negative_samples = pd.DataFrame(
    matched_negative_samples
)

print("Shape:", matched_negative_samples.shape)

print("\nUnique reference dates:",
      matched_negative_samples["reference_date"].nunique())

print("\nTarget:")
print(matched_negative_samples["LANDSLIDE_NEXT_3D"].value_counts())

print("\nUnique negative locations:",
      matched_negative_samples[
          ["latitude", "longitude"]
      ].drop_duplicates().shape[0])

print("\nSample:")
print(matched_negative_samples.head())

Shape: (377, 4)

Unique reference dates: 212

Target:
LANDSLIDE_NEXT_3D
0    377
Name: count, dtype: int64

Unique negative locations: 229

Sample:
    latitude  longitude reference_date  LANDSLIDE_NEXT_3D
0  31.466509  76.818404     2011-06-26                  0
1  29.969322  78.168015     2011-07-06                  0
2  30.245145  78.457712     2011-07-06                  0
3  31.554625  78.251096     2011-07-06                  0
4  32.568286  77.103797     2011-07-18                  0


In [63]:
matched_negative_records = []

for i, row in matched_negative_samples.iterrows():

    values = get_antecedent_rainfall(
        float(row["latitude"]),
        float(row["longitude"]),
        row["reference_date"].strftime("%Y-%m-%d")
    ).getInfo()

    matched_negative_records.append({
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "reference_date": row["reference_date"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D"),
        "LANDSLIDE_NEXT_3D": 0
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} / {len(matched_negative_samples)}")

matched_negative_temporal = pd.DataFrame(
    matched_negative_records
)

matched_negative_temporal.to_csv(
    "../data/matched_negative_temporal.csv",
    index=False
)

print("\nSaved:", matched_negative_temporal.shape)
print(matched_negative_temporal.head())

Processed 25 / 377
Processed 50 / 377
Processed 75 / 377
Processed 100 / 377
Processed 125 / 377
Processed 150 / 377
Processed 175 / 377
Processed 200 / 377
Processed 225 / 377
Processed 250 / 377
Processed 275 / 377
Processed 300 / 377
Processed 325 / 377
Processed 350 / 377
Processed 375 / 377

Saved: (377, 9)
    latitude  longitude reference_date    RAIN_1D    RAIN_3D    RAIN_7D  \
0  31.466509  76.818404     2011-06-26  53.709633  71.612843  71.612843   
1  29.969322  78.168015     2011-07-06   0.000000   0.000000  25.924295   
2  30.245145  78.457712     2011-07-06  26.038486  26.038486  26.038486   
3  31.554625  78.251096     2011-07-06   0.000000   0.000000   9.850498   
4  32.568286  77.103797     2011-07-18   0.000000   0.000000   0.000000   

     RAIN_15D    RAIN_30D  LANDSLIDE_NEXT_3D  
0  111.228046  187.958744                  0  
1  170.707731  233.244482                  0  
2  150.454201  207.811710                  0  
3   56.391357   77.528316                  0  


In [64]:
print("Shape:", matched_negative_temporal.shape)

print("\nMissing values:")
print(matched_negative_temporal.isnull().sum())

print("\nTarget distribution:")
print(matched_negative_temporal["LANDSLIDE_NEXT_3D"].value_counts())

print("\nReference date range:")
print(
    matched_negative_temporal["reference_date"].min(),
    "to",
    matched_negative_temporal["reference_date"].max()
)

print("\nRainfall summary:")
print(
    matched_negative_temporal[
        ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
    ].describe()
)

Shape: (377, 9)

Missing values:
latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
LANDSLIDE_NEXT_3D    0
dtype: int64

Target distribution:
LANDSLIDE_NEXT_3D
0    377
Name: count, dtype: int64

Reference date range:
2007-07-27 00:00:00 to 2018-10-06 00:00:00

Rainfall summary:
          RAIN_1D     RAIN_3D     RAIN_7D    RAIN_15D    RAIN_30D
count  377.000000  377.000000  377.000000  377.000000  377.000000
mean    10.483166   33.837862   72.728361  155.581393  284.860038
std     23.601823   45.335091   71.831630  118.331040  200.943245
min      0.000000    0.000000    0.000000    0.000000    0.000000
25%      0.000000    0.000000   17.097187   57.573708  108.287041
50%      0.000000   16.495651   55.649185  136.850876  254.659630
75%      9.679436   53.258350  109.734985  230.083368  430.945500
max    202.472549  331.530075  598.614166  598.614166  983.

In [65]:
from sklearn.neighbors import NearestNeighbors

# Static dataset
static_df = pd.read_csv(
    "../data/himalayan_landslide_v1_dataset.csv"
)

# Coordinates of static dataset
static_coords = static_df[
    ["LATITUDE", "LONGITUDE"]
].values

# Coordinates of negative samples
negative_coords = matched_negative_temporal[
    ["latitude", "longitude"]
].values

# Nearest static point
nn = NearestNeighbors(n_neighbors=1)
nn.fit(static_coords)

distances, indices = nn.kneighbors(negative_coords)

# Attach static features
nearest_static = static_df.iloc[
    indices[:, 0]
][
    ["ELEVATION", "SLOPE", "LANDCOVER_CODE"]
].reset_index(drop=True)

matched_negative_temporal = pd.concat(
    [
        matched_negative_temporal.reset_index(drop=True),
        nearest_static
    ],
    axis=1
)

print("Shape:", matched_negative_temporal.shape)

print("\nMissing values:")
print(matched_negative_temporal.isnull().sum())

print("\nFinal columns:")
print(matched_negative_temporal.columns.tolist())

print("\nSample:")
print(matched_negative_temporal.head())

Shape: (377, 12)

Missing values:
latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
LANDSLIDE_NEXT_3D    0
ELEVATION            0
SLOPE                0
LANDCOVER_CODE       0
dtype: int64

Final columns:
['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE_NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']

Sample:
    latitude  longitude reference_date    RAIN_1D    RAIN_3D    RAIN_7D  \
0  31.466509  76.818404     2011-06-26  53.709633  71.612843  71.612843   
1  29.969322  78.168015     2011-07-06   0.000000   0.000000  25.924295   
2  30.245145  78.457712     2011-07-06  26.038486  26.038486  26.038486   
3  31.554625  78.251096     2011-07-06   0.000000   0.000000   9.850498   
4  32.568286  77.103797     2011-07-18   0.000000   0.000000   0.000000   

     RAIN_15D    RAIN_30D  LANDSLIDE_NE

In [66]:
# Convert coordinate differences to approximate km
lat_diff = distances[:, 0] * 111.0
lon_diff = (
    distances[:, 0]
    * 111.0
    * np.cos(np.radians(negative_coords[:, 0]))
)

approx_distance_km = np.sqrt(
    lat_diff**2 + lon_diff**2
)

print("Nearest static-point distance (km):")
print(
    pd.Series(approx_distance_km).describe()
)

print("\nMaximum distance:", approx_distance_km.max(), "km")
print("Median distance:", np.median(approx_distance_km), "km")


Nearest static-point distance (km):
count    377.000000
mean       1.821455
std        1.067895
min        0.000000
25%        0.972833
50%        1.767022
75%        2.417939
max        6.061087
dtype: float64

Maximum distance: 6.061087085005026 km
Median distance: 1.767022235720458 km


In [67]:
# Prepare positive dataset
positive_final = positive_temporal[
    [
        "latitude",
        "longitude",
        "reference_date",
        "RAIN_1D",
        "RAIN_3D",
        "RAIN_7D",
        "RAIN_15D",
        "RAIN_30D",
        "ELEVATION",
        "SLOPE",
        "LANDCOVER_CODE",
        "LANDSLIDE_NEXT_3D"
    ]
].copy()

# Prepare negative dataset
negative_final = matched_negative_temporal[
    [
        "latitude",
        "longitude",
        "reference_date",
        "RAIN_1D",
        "RAIN_3D",
        "RAIN_7D",
        "RAIN_15D",
        "RAIN_30D",
        "ELEVATION",
        "SLOPE",
        "LANDCOVER_CODE",
        "LANDSLIDE_NEXT_3D"
    ]
].copy()

# Combine
temporal_dataset_v2 = pd.concat(
    [positive_final, negative_final],
    ignore_index=True
)

# Shuffle reproducibly
temporal_dataset_v2 = temporal_dataset_v2.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Save
temporal_dataset_v2.to_csv(
    "../data/himalayan_temporal_dataset_v2.csv",
    index=False
)

print("Shape:", temporal_dataset_v2.shape)

print("\nTarget distribution:")
print(
    temporal_dataset_v2["LANDSLIDE_NEXT_3D"].value_counts()
)

print("\nMissing values:")
print(
    temporal_dataset_v2.isnull().sum()
)

Shape: (754, 12)

Target distribution:
LANDSLIDE_NEXT_3D
0    377
1    377
Name: count, dtype: int64

Missing values:
latitude             0
longitude            0
reference_date       0
RAIN_1D              0
RAIN_3D              0
RAIN_7D              0
RAIN_15D             0
RAIN_30D             0
ELEVATION            0
SLOPE                0
LANDCOVER_CODE       0
LANDSLIDE_NEXT_3D    0
dtype: int64


In [68]:
# Calculate days from reference date to the recorded event
positive_target_check = positive_samples[
    ["latitude", "longitude", "reference_date", "event_date"]
].copy()

positive_target_check["reference_date"] = pd.to_datetime(
    positive_target_check["reference_date"]
).dt.normalize()

positive_target_check["event_date"] = pd.to_datetime(
    positive_target_check["event_date"]
).dt.normalize()

positive_target_check["days_to_event"] = (
    positive_target_check["event_date"]
    - positive_target_check["reference_date"]
).dt.days

print(
    positive_target_check["days_to_event"].value_counts()
    .sort_index()
)

days_to_event
3    377
Name: count, dtype: int64


In [69]:
event_history = (
    event_dataset[
        ["latitude", "longitude", "event_date"]
    ].copy()
)

event_history["event_date"] = pd.to_datetime(
    event_history["event_date"]
).dt.normalize()

event_history = event_history.drop_duplicates()

location_event_counts = (
    event_history
    .groupby(["latitude", "longitude"])
    .size()
)

print("Total event records:", len(event_history))
print("Unique event locations:", len(location_event_counts))

print("\nLocations with repeated events:")
print(
    location_event_counts[
        location_event_counts > 1
    ].sort_values(ascending=False).head(20)
)

print("\nMaximum events at one location:",
      location_event_counts.max())

Total event records: 375
Unique event locations: 374

Locations with repeated events:
latitude  longitude
30.6738   77.3544      2
dtype: int64

Maximum events at one location: 2


In [70]:
# Create positive samples for all three forecast horizons

event_history = event_history.sort_values(
    ["latitude", "longitude", "event_date"]
).reset_index(drop=True)

multi_horizon_positive = []

for _, row in event_history.iterrows():

    lat = row["latitude"]
    lon = row["longitude"]
    event_date = row["event_date"]

    # Reference date 1 day before event
    multi_horizon_positive.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": event_date - pd.Timedelta(days=1),
        "event_date": event_date,
        "positive_horizon": 1
    })

    # Reference date 2 days before event
    multi_horizon_positive.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": event_date - pd.Timedelta(days=2),
        "event_date": event_date,
        "positive_horizon": 2
    })

    # Reference date 3 days before event
    multi_horizon_positive.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": event_date - pd.Timedelta(days=3),
        "event_date": event_date,
        "positive_horizon": 3
    })

multi_horizon_positive = pd.DataFrame(
    multi_horizon_positive
)

print("Shape:", multi_horizon_positive.shape)

print("\nSamples by horizon:")
print(
    multi_horizon_positive["positive_horizon"]
    .value_counts()
    .sort_index()
)

print("\nSample:")
print(multi_horizon_positive.head(10))

Shape: (1125, 5)

Samples by horizon:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Sample:
    latitude  longitude reference_date event_date  positive_horizon
0  28.866500  79.568200     2011-08-11 2011-08-12                 1
1  28.866500  79.568200     2011-08-10 2011-08-12                 2
2  28.866500  79.568200     2011-08-09 2011-08-12                 3
3  29.158500  79.146400     2014-07-22 2014-07-23                 1
4  29.158500  79.146400     2014-07-21 2014-07-23                 2
5  29.158500  79.146400     2014-07-20 2014-07-23                 3
6  29.295586  79.540431     2010-09-17 2010-09-18                 1
7  29.295586  79.540431     2010-09-16 2010-09-18                 2
8  29.295586  79.540431     2010-09-15 2010-09-18                 3
9  29.330000  80.100000     2008-07-17 2008-07-18                 1


In [71]:
# Check whether each generated positive sample has
# another event between its reference date and target horizon.

location_events = (
    event_history
    .groupby(["latitude", "longitude"])["event_date"]
    .apply(list)
    .to_dict()
)

checks = []

for _, row in multi_horizon_positive.iterrows():

    key = (row["latitude"], row["longitude"])
    ref_date = row["reference_date"]
    horizon = row["positive_horizon"]

    future_events = [
        d for d in location_events[key]
        if ref_date < d <= ref_date + pd.Timedelta(days=horizon)
    ]

    checks.append(len(future_events))

multi_horizon_positive["future_event_count"] = checks

print(
    multi_horizon_positive["future_event_count"]
    .value_counts()
    .sort_index()
)

print("\nSamples with more than one future event:")
print(
    (multi_horizon_positive["future_event_count"] > 1).sum()
)

future_event_count
1    1125
Name: count, dtype: int64

Samples with more than one future event:
0


In [72]:
multi_positive_records = []

for i, row in multi_horizon_positive.iterrows():

    values = get_antecedent_rainfall(
        float(row["latitude"]),
        float(row["longitude"]),
        row["reference_date"].strftime("%Y-%m-%d")
    ).getInfo()

    multi_positive_records.append({
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "reference_date": row["reference_date"],
        "event_date": row["event_date"],
        "positive_horizon": row["positive_horizon"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D"),
        "LANDSLIDE": 1
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} / {len(multi_horizon_positive)}")

multi_positive_temporal = pd.DataFrame(
    multi_positive_records
)

multi_positive_temporal.to_csv(
    "../data/multi_horizon_positive_temporal.csv",
    index=False
)

print("\nSaved:", multi_positive_temporal.shape)
print(multi_positive_temporal.head())

Processed 25 / 1125
Processed 50 / 1125
Processed 75 / 1125
Processed 100 / 1125
Processed 125 / 1125
Processed 150 / 1125
Processed 175 / 1125
Processed 200 / 1125
Processed 225 / 1125
Processed 250 / 1125
Processed 275 / 1125
Processed 300 / 1125
Processed 325 / 1125
Processed 350 / 1125
Processed 375 / 1125
Processed 400 / 1125
Processed 425 / 1125
Processed 450 / 1125
Processed 475 / 1125
Processed 500 / 1125
Processed 525 / 1125
Processed 550 / 1125
Processed 575 / 1125
Processed 600 / 1125
Processed 625 / 1125
Processed 650 / 1125
Processed 675 / 1125
Processed 700 / 1125
Processed 725 / 1125
Processed 750 / 1125
Processed 775 / 1125
Processed 800 / 1125
Processed 825 / 1125
Processed 850 / 1125
Processed 875 / 1125
Processed 900 / 1125
Processed 925 / 1125
Processed 950 / 1125
Processed 975 / 1125
Processed 1000 / 1125
Processed 1025 / 1125
Processed 1050 / 1125
Processed 1075 / 1125
Processed 1100 / 1125
Processed 1125 / 1125

Saved: (1125, 11)
   latitude  longitude reference_

In [73]:
multi_horizon_positive.to_csv(
    "../data/multi_horizon_positive_definitions.csv",
    index=False
)

print("Saved successfully")
print(multi_horizon_positive.shape)

Saved successfully
(1125, 6)


In [74]:
# Save the important datasets currently in memory

positive_temporal.to_csv(
    "../data/positive_temporal_samples.csv",
    index=False
)

matched_negative_temporal.to_csv(
    "../data/matched_negative_temporal.csv",
    index=False
)

temporal_dataset_v2.to_csv(
    "../data/himalayan_temporal_dataset_v2.csv",
    index=False
)

multi_horizon_positive.to_csv(
    "../data/multi_horizon_positive_definitions.csv",
    index=False
)

print("All current datasets saved.")

All current datasets saved.


In [75]:
import os

files = [
    "../data/positive_temporal_samples.csv",
    "../data/matched_negative_temporal.csv",
    "../data/himalayan_temporal_dataset_v2.csv",
    "../data/multi_horizon_positive_definitions.csv"
]

for f in files:
    print(f, "->", os.path.exists(f))

../data/positive_temporal_samples.csv -> True
../data/matched_negative_temporal.csv -> True
../data/himalayan_temporal_dataset_v2.csv -> True
../data/multi_horizon_positive_definitions.csv -> True


In [1]:
import ee

ee.Initialize(project="integral-server-484809-p2")

imerg = ee.ImageCollection("NASA/GPM_L3/IMERG_V07")

# Use a sample already present in your notebook
lat = 28.8665
lon = 79.5682
reference_date = "2011-08-11"

point = ee.Geometry.Point([lon, lat])
end = ee.Date(reference_date)

windows = [1, 3, 7, 15, 30]
rainfall_images = []
image_counts = {}

for days in windows:
    collection = imerg.filterDate(
        end.advance(-days, "day"),
        end
    )

    image_counts[days] = collection.size().getInfo()

    # IMERG is a rainfall rate in mm/hour.
    # Each image represents a 30-minute interval.
    rainfall = (
        collection
        .select("precipitation")
        .map(lambda image: image.multiply(0.5))
        .sum()
        .rename(f"RAIN_{days}D")
    )

    rainfall_images.append(rainfall)

combined = ee.Image.cat(rainfall_images)

values = combined.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=11132,
    maxPixels=1_000_000
).getInfo()

print("Reference date:", reference_date)
print("Image counts:", image_counts)
print("IMERG rainfall totals (mm):", values)

Reference date: 2011-08-11
Image counts: {1: 48, 3: 144, 7: 336, 15: 720, 30: 1440}
IMERG rainfall totals (mm): {'RAIN_15D': 218.90499574504793, 'RAIN_1D': 0.2199999988079071, 'RAIN_30D': 492.6599887367338, 'RAIN_3D': 0.3349999934434891, 'RAIN_7D': 64.39499890804291}
